# Engenharia de Features (v6) — Corrigida
## v4 + Histórico de Copas (com correção de pênaltis e 3º lugar)

**Correções aplicadas em relação à v6 original:**
- **Bug 1 — Pênaltis:** A Argentina era classificada como vice-campeã (fase 5) porque a final contra a França terminou 3x3. Agora usamos `shootouts.csv` para determinar o vencedor real → Argentina = fase 6 (campeã)
- **Bug 2 — 3º lugar:** A Croácia era classificada como campeã (fase 6) porque venceu o jogo do 3º lugar. Agora verificamos se o time perdeu a semifinal antes de classificar o 7º jogo → Croácia = fase 4 (semifinal)

**Dataset de saída:** `data/processed/features_completo_v6.csv`

## 1. Imports e Carregamento

In [18]:
import pandas as pd
import numpy as np
import sys
sys.path.append('../src/features')
from elo import calcular_elo_historico

df_raw       = pd.read_csv('../data/raw/results.csv', parse_dates=['date'])
df_shootouts = pd.read_csv('../data/raw/shootouts.csv', parse_dates=['date'])

print(f'Resultados: {df_raw.shape}')
print(f'Shootouts (pênaltis): {df_shootouts.shape}')
print(df_shootouts.head(3))

Resultados: (49287, 9)
Shootouts (pênaltis): (675, 5)
        date    home_team         away_team       winner first_shooter
0 1967-08-22        India            Taiwan       Taiwan           NaN
1 1971-11-14  South Korea  Vietnam Republic  South Korea           NaN
2 1972-05-07  South Korea              Iraq         Iraq           NaN


## 2. Calculando ELO Histórico

In [19]:
df = calcular_elo_historico(df_raw, elo_inicial=1000, k_competitivo=40, k_amistoso=20)
print(f'ELO calculado para {len(df)} jogos')

ELO calculado para 49287 jogos


## 3. Funções de Histórico — Versão Corrigida

### Codificação de fase:
- 0 = não classificou
- 1 = fase de grupos
- 2 = oitavas
- 3 = quartas
- 4 = semifinal (inclui 3º lugar)
- 5 = final/vice
- 6 = campeão

In [20]:
copas_datas = {
    1990: ('1990-06-08', '1990-07-08'),
    1994: ('1994-06-17', '1994-07-17'),
    1998: ('1998-06-10', '1998-07-12'),
    2002: ('2002-05-31', '2002-06-30'),
    2006: ('2006-06-09', '2006-07-09'),
    2010: ('2010-06-11', '2010-07-11'),
    2014: ('2014-06-12', '2014-07-13'),
    2018: ('2018-06-14', '2018-07-15'),
    2022: ('2022-11-20', '2022-12-18'),
}

def venceu_jogo(row, selecao, df_shootouts):
    """Verifica se a seleção venceu um jogo, incluindo pênaltis."""
    home = row['home_team'] == selecao
    gs = row['home_score'] if home else row['away_score']
    gc = row['away_score'] if home else row['home_score']

    if gs > gc:
        return True
    elif gc > gs:
        return False
    else:
        # Empate — verificar pênaltis
        penalty = df_shootouts[
            (df_shootouts['date'] == row['date']) &
            (
                ((df_shootouts['home_team'] == row['home_team']) &
                 (df_shootouts['away_team'] == row['away_team'])) |
                ((df_shootouts['home_team'] == row['away_team']) &
                 (df_shootouts['away_team'] == row['home_team']))
            )
        ]
        if len(penalty) > 0:
            return penalty.iloc[0]['winner'] == selecao
        return False


def get_fase(df_copa, selecao, df_shootouts):
    """
    Determina a fase atingida por uma seleção na Copa.
    Corrige dois bugs da versão anterior:
    1. Pênaltis contados corretamente via shootouts.csv
    2. 3º lugar não confundido com final
    """
    jogos = df_copa[
        (df_copa['home_team'] == selecao) |
        (df_copa['away_team'] == selecao)
    ].sort_values('date')

    if len(jogos) == 0: return 0

    n = len(jogos)

    if n <= 3: return 1   # grupos
    elif n == 4: return 2  # oitavas
    elif n == 5: return 3  # quartas
    elif n == 6: return 4  # semi (perdeu e não jogou 3º lugar, ou jogou 3º com 6 jogos)
    elif n == 7:
        # 6º jogo = semifinal
        jogo_semi = jogos.iloc[5]
        perdeu_semi = not venceu_jogo(jogo_semi, selecao, df_shootouts)

        if perdeu_semi:
            return 4  # foi ao 3º lugar mas fase máxima = semi

        # Chegou à final — 7º jogo
        jogo_final = jogos.iloc[6]
        if venceu_jogo(jogo_final, selecao, df_shootouts):
            return 6  # campeão
        else:
            return 5  # vice
    return min(n - 2, 6)


def get_media_gols_copa(df_copa, selecao):
    jogos = df_copa[
        (df_copa['home_team'] == selecao) |
        (df_copa['away_team'] == selecao)
    ]
    if len(jogos) == 0: return np.nan
    return np.mean([
        row['home_score'] if row['home_team'] == selecao else row['away_score']
        for _, row in jogos.iterrows()
    ])

print('Funções definidas!')

Funções definidas!


## 4. Validação das Correções

Verificamos todos os casos problemáticos da versão anterior.

In [21]:
copa_2022 = df[
    (df['tournament'] == 'FIFA World Cup') &
    (df['date'] >= '2022-11-20') & (df['date'] <= '2022-12-18')
]

testes = [
    ('Argentina',    6, 'Campeã 2022 (pênaltis vs França)'),
    ('France',       5, 'Vice 2022'),
    ('Croatia',      4, '3º lugar 2022 — fase máxima = semi'),
    ('Morocco',      4, '4º lugar 2022 — perdeu semi'),
    ('Brazil',       3, 'Quartas 2022'),
    ('Netherlands',  3, 'Quartas 2022'),
    ('England',      3, 'Quartas 2022'),
    ('Portugal',     3, 'Quartas 2022'),
    ('Spain',        2, 'Oitavas 2022'),
    ('Belgium',      1, 'Grupos 2022'),
    ('Germany',      1, 'Grupos 2022'),
    ('Japan',        2, 'Oitavas 2022 (pênaltis vs Croácia)'),
    ('Norway',       0, 'Não classificada 2022'),
    ('Colombia',     0, 'Não classificada 2022'),
]

erros = 0
print(f'{"Seleção":<15} {"Esperado":>8} {"Calculado":>10} {"Status"}')
print('-' * 50)
for selecao, esperado, descricao in testes:
    resultado = get_fase(copa_2022, selecao, df_shootouts)
    ok = resultado == esperado
    if not ok: erros += 1
    status = '✅' if ok else f'❌ (esperado {esperado})'
    print(f'{selecao:<15} {esperado:>8} {resultado:>10} {status}  {descricao}')

print(f'\n{"✅ Todos corretos!" if erros == 0 else f"❌ {erros} erro(s)"}')

Seleção         Esperado  Calculado Status
--------------------------------------------------
Argentina              6          6 ✅  Campeã 2022 (pênaltis vs França)
France                 5          5 ✅  Vice 2022
Croatia                4          4 ✅  3º lugar 2022 — fase máxima = semi
Morocco                4          4 ✅  4º lugar 2022 — perdeu semi
Brazil                 3          3 ✅  Quartas 2022
Netherlands            3          3 ✅  Quartas 2022
England                3          3 ✅  Quartas 2022
Portugal               3          3 ✅  Quartas 2022
Spain                  2          2 ✅  Oitavas 2022
Belgium                1          1 ✅  Grupos 2022
Germany                1          1 ✅  Grupos 2022
Japan                  2          2 ✅  Oitavas 2022 (pênaltis vs Croácia)
Norway                 0          0 ✅  Não classificada 2022
Colombia               0          0 ✅  Não classificada 2022

✅ Todos corretos!


## 5. Função de Features v6

In [22]:
def calcular_features_v6(selecao, ciclo, copa_df, copa_ano, min_jogos=15):
    jogos = ciclo[
        (ciclo['home_team'] == selecao) |
        (ciclo['away_team'] == selecao)
    ].sort_values('date')

    if len(jogos) < min_jogos:
        raise ValueError(f'Apenas {len(jogos)} jogos')

    gm, gs, vit, elo_adv = [], [], [], []
    for _, row in jogos.iterrows():
        if row['home_team'] == selecao:
            gm.append(row['home_score']); gs.append(row['away_score'])
            vit.append(1 if row['home_score'] > row['away_score'] else 0)
            elo_adv.append(row['elo_away_antes'])
        else:
            gm.append(row['away_score']); gs.append(row['home_score'])
            vit.append(1 if row['away_score'] > row['home_score'] else 0)
            elo_adv.append(row['elo_home_antes'])

    gm, gs, vit = np.array(gm), np.array(gs), np.array(vit)
    elo_adv = np.array(elo_adv)

    ult15 = jogos.tail(15)
    gm15, gs15, vit15, ea15 = [], [], [], []
    for _, row in ult15.iterrows():
        if row['home_team'] == selecao:
            gm15.append(row['home_score']); gs15.append(row['away_score'])
            vit15.append(1 if row['home_score'] > row['away_score'] else 0)
            ea15.append(row['elo_away_antes'])
        else:
            gm15.append(row['away_score']); gs15.append(row['home_score'])
            vit15.append(1 if row['away_score'] > row['home_score'] else 0)
            ea15.append(row['elo_home_antes'])

    # Target
    sel_copa = copa_df[
        (copa_df['home_team'] == selecao) |
        (copa_df['away_team'] == selecao)
    ]
    gols_copa = [
        row['home_score'] if row['home_team'] == selecao else row['away_score']
        for _, row in sel_copa.iterrows()
    ]

    # Histórico de Copas
    anos_copas  = sorted(copas_datas.keys())
    idx_atual   = anos_copas.index(copa_ano)
    copas_ant   = anos_copas[max(0, idx_atual-2):idx_atual]

    gols_hist = []
    for ano_h in copas_ant:
        ini, fim = copas_datas[ano_h]
        copa_h = df[
            (df['tournament'] == 'FIFA World Cup') &
            (df['date'] >= ini) & (df['date'] <= fim)
        ]
        mg = get_media_gols_copa(copa_h, selecao)
        if not np.isnan(mg):
            gols_hist.append(mg)
    media_hist = np.mean(gols_hist) if gols_hist else 0.0

    # Fase da Copa anterior
    if idx_atual > 0:
        ano_ant = anos_copas[idx_atual - 1]
        ini_ant, fim_ant = copas_datas[ano_ant]
        copa_ant_df = df[
            (df['tournament'] == 'FIFA World Cup') &
            (df['date'] >= ini_ant) & (df['date'] <= fim_ant)
        ]
        fase_ant = get_fase(copa_ant_df, selecao, df_shootouts)
    else:
        fase_ant = 0

    return {
        'media_gols_marcados_ciclo': gm.mean(),
        'media_gols_sofridos_ciclo': gs.mean(),
        'pct_vitorias_ciclo':        vit.mean(),
        'total_jogos_ciclo':         len(jogos),
        'media_gols_marcados_ult15': np.array(gm15).mean(),
        'media_gols_sofridos_ult15': np.array(gs15).mean(),
        'pct_vitorias_ult15':        np.array(vit15).mean(),
        'elo_medio_adv_ciclo':       elo_adv.mean(),
        'elo_medio_adv_ult15':       np.array(ea15).mean(),
        'media_gols_ultimas2_copas': media_hist,
        'fase_ultima_copa':          fase_ant,
        'media_gols_copa':           np.mean(gols_copa)
    }

print('Função v6 definida!')

Função v6 definida!


## 6. Pipeline Completo — Todas as Copas (1994–2022)

In [23]:
copas = {
    1994: {'ciclo_inicio': '1990-07-09', 'ciclo_fim': '1994-06-16', 'copa_inicio': '1994-06-17', 'copa_fim': '1994-07-17'},
    1998: {'ciclo_inicio': '1994-07-18', 'ciclo_fim': '1998-06-09', 'copa_inicio': '1998-06-10', 'copa_fim': '1998-07-12'},
    2002: {'ciclo_inicio': '1998-07-13', 'ciclo_fim': '2002-05-30', 'copa_inicio': '2002-05-31', 'copa_fim': '2002-06-30'},
    2006: {'ciclo_inicio': '2002-07-01', 'ciclo_fim': '2006-06-08', 'copa_inicio': '2006-06-09', 'copa_fim': '2006-07-09'},
    2010: {'ciclo_inicio': '2006-07-10', 'ciclo_fim': '2010-06-10', 'copa_inicio': '2010-06-11', 'copa_fim': '2010-07-11'},
    2014: {'ciclo_inicio': '2010-07-12', 'ciclo_fim': '2014-06-11', 'copa_inicio': '2014-06-12', 'copa_fim': '2014-07-13'},
    2018: {'ciclo_inicio': '2014-07-14', 'ciclo_fim': '2018-06-13', 'copa_inicio': '2018-06-14', 'copa_fim': '2018-07-15'},
    2022: {'ciclo_inicio': '2018-07-16', 'ciclo_fim': '2022-11-19', 'copa_inicio': '2022-11-20', 'copa_fim': '2022-12-18'},
}

todos_dados = []
filtrados   = []

for ano, datas in copas.items():
    print(f'Processando Copa {ano}...')

    ciclo = df[
        (df['date'] >= datas['ciclo_inicio']) &
        (df['date'] <= datas['ciclo_fim']) &
        (df['tournament'] != 'FIFA World Cup')
    ]
    copa_df = df[
        (df['tournament'] == 'FIFA World Cup') &
        (df['date'] >= datas['copa_inicio']) &
        (df['date'] <= datas['copa_fim'])
    ]
    selecoes = pd.unique(copa_df[['home_team', 'away_team']].values.ravel())

    for selecao in selecoes:
        try:
            resultado = calcular_features_v6(selecao, ciclo, copa_df, ano)
            resultado['selecao']   = selecao
            resultado['copa_alvo'] = ano
            todos_dados.append(resultado)
        except ValueError as e:
            filtrados.append({'selecao': selecao, 'copa': ano, 'motivo': str(e)})
        except Exception as e:
            print(f'  Erro em {selecao}: {e}')

df_v6 = pd.DataFrame(todos_dados)
print(f'\nDataset v6: {df_v6.shape[0]} linhas × {df_v6.shape[1]} colunas')
print(f'Filtrados: {len(filtrados)}')

# Verificar Argentina e Croácia em 2022
print('\nVerificação Copa 2022:')
check = df_v6[df_v6['copa_alvo'] == 2022][['selecao', 'fase_ultima_copa', 'media_gols_copa']]
check_sel = check[check['selecao'].isin(['Argentina', 'Croatia', 'France', 'Morocco', 'Brazil'])]
print(check_sel.to_string(index=False))

Processando Copa 1994...
Processando Copa 1998...
Processando Copa 2002...
Processando Copa 2006...
Processando Copa 2010...
Processando Copa 2014...
Processando Copa 2018...
Processando Copa 2022...

Dataset v6: 248 linhas × 14 colunas
Filtrados: 0

Verificação Copa 2022:
  selecao  fase_ultima_copa  media_gols_copa
Argentina                 2         2.142857
   France                 6         2.285714
  Morocco                 1         0.857143
  Croatia                 5         1.142857
   Brazil                 3         1.600000


## 7. Salvamento

In [24]:
df_v6.to_csv('../data/processed/features_completo_v6.csv', index=False)
print('Dataset v6 (corrigido) salvo em: data/processed/features_completo_v6.csv')
print(f'\nLinhas por Copa:')
print(df_v6['copa_alvo'].value_counts().sort_index())

Dataset v6 (corrigido) salvo em: data/processed/features_completo_v6.csv

Linhas por Copa:
copa_alvo
1994    24
1998    32
2002    32
2006    32
2010    32
2014    32
2018    32
2022    32
Name: count, dtype: int64
